# Exploring Weather Data

In [4]:
import pandas as pd

In [5]:
incident_data = pd.read_csv("../../clean_data/florida_incident_data_w_categories.csv")

In [6]:
is_valid_incident = incident_data["incident_type"] != "Other (None of the above)"

In [7]:
is_valid_incident.value_counts()

incident_type
False    8522
True     1459
Name: count, dtype: int64

In [8]:
incident_data[is_valid_incident]["ID"].nunique()

1459

In [21]:
from get_weather import get_weather_data
from datetime import datetime

def get_weather_for_incident(row):

  date_str = row["EventDate"]
  latitude, longitude = row["Latitude"], row["Longitude"]

  if latitude > 90 or latitude < -90 or pd.isna(latitude):
    return {}

  if pd.isna(longitude):
      return {}

  date_converted = pd.to_datetime(date_str, format="%Y-%m-%d")
  weather_data = get_weather_data(latitude, longitude, date_converted)
  data = weather_data.to_dict(orient="records")[0]
  data["associated_incident_id"] = row["ID"]
  return data


In [22]:
valid_incident_data = incident_data[is_valid_incident]

In [26]:
from tqdm.auto import tqdm

# 1. Register tqdm with pandas
tqdm.pandas(desc="Processing rows")

In [ ]:
results = valid_incident_data.progress_apply(get_weather_for_incident, axis=1, result_type="expand")

Processing rows: 100%|██████████| 1459/1459 [00:04<00:00, 294.29it/s]


In [29]:
results.to_csv("valid_incident_weather_info.csv", index=False)

In [58]:
import random
from datetime import timedelta

def get_weather_for_nonincident(row):

  id = row["ID"]
  date_str = row["EventDate"]
  latitude, longitude = row["Latitude"], row["Longitude"]

  if latitude > 90 or latitude < -90 or pd.isna(latitude):
    return {}

  if pd.isna(longitude):
      return {}

  date_converted = pd.to_datetime(date_str, format="%Y-%m-%d")

  random.seed(int(id))
  date_converted = date_converted + timedelta(days=random.randint(-30, 30))
  
  weather_data = get_weather_data(latitude, longitude, date_converted)
  data = weather_data.to_dict(orient="records")[0]
  data["associated_incident_id"] = id
  return data


In [59]:
get_weather_for_nonincident(valid_incident_data.iloc[0])

{'date': Timestamp('2025-08-17 00:00:00+0000', tz='UTC'),
 'weather_code': 63.0,
 'temperature_2m_mean': 27.097915649414062,
 'temperature_2m_max': 31.989213943481445,
 'temperature_2m_min': 24.08043670654297,
 'apparent_temperature_mean': 32.90245056152344,
 'apparent_temperature_max': 39.26896667480469,
 'apparent_temperature_min': 29.33065414428711,
 'precipitation_sum': 6.400000095367432,
 'rain_sum': 6.400000095367432,
 'snowfall_sum': 0.0,
 'precipitation_hours': 8.0,
 'wind_speed_10m_max': 9.756597518920898,
 'wind_gusts_10m_max': 18.0,
 'shortwave_radiation_sum': 18.670000076293945,
 'et0_fao_evapotranspiration': 3.945141553878784,
 'relative_humidity_2m_mean': 86.25646209716797,
 'relative_humidity_2m_max': 96.45608520507812,
 'relative_humidity_2m_min': 65.00041198730469,
 'soil_moisture_0_to_100cm_mean': 0.10071500390768051,
 'soil_temperature_0_to_100cm_mean': 28.012962341308594,
 'associated_incident_id': np.int64(2025098920)}

In [62]:
random_non_incident_data = valid_incident_data.progress_apply(get_weather_for_nonincident, axis=1, result_type="expand")

Processing rows: 100%|██████████| 1459/1459 [00:28<00:00, 51.83it/s] 


In [64]:
random_non_incident_data.to_csv("invalid_incident_weather_info.csv", index=False)